<a href="https://colab.research.google.com/github/zara-nasiruddin/computational-art-history/blob/main/project2_clip_prompt_robustness.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:

# Pairwise agreement between prompt conditions
agreement = controlled_pivot.apply(
    lambda col: controlled_pivot.eq(col, axis=0).mean()
)

print("=== PROMPT-TO-PROMPT PREDICTION AGREEMENT ===")
display(agreement.round(2))

# Per-image robustness: number of distinct labels across prompts
print("\n=== IMAGE-LEVEL STABILITY ===")
print(
    controlled_pivot.nunique(axis=1)
    .value_counts()
    .sort_index()
    .rename("number_of_images")
    .to_string()
)

# Artist-level prediction distribution for each controlled prompt
print("\n=== PREDICTIONS BY ARTIST ===")
artist_counts = controlled_df.pivot_table(
    index="artist",
    columns=["template", "prediction"],
    values="image_id",
    aggfunc="count",
    fill_value=0
)
display(artist_counts)

agreement.to_csv("project2_prompt_agreement.csv")
print("\nSaved project2_prompt_agreement.csv")


=== PROMPT-TO-PROMPT PREDICTION AGREEMENT ===


template,artistic_style,artwork,characteristic,noun,style
template,,,,,
artistic_style,1.00,0.25,0.92,0.00,0.45
artwork,0.25,1.00,0.28,0.75,0.75
characteristic,0.92,0.28,1.00,0.08,0.52
noun,0.00,0.75,0.08,1.00,0.55
style,0.45,0.75,0.52,0.55,1.00



=== IMAGE-LEVEL STABILITY ===
2    40

=== PREDICTIONS BY ARTIST ===


template             artistic_style          artwork  \
prediction         high_renaissance high_renaissance   
artist                                                 
agnolo-bronzino                  18                5   
andrea-del-sarto                 10                2   
andrea-mantegna                   6                2   
andrea-solario                    2                1   
cima-da-conegliano                2                0   
correggio                         2                0   

template                                        characteristic  \
prediction         mannerism_late_renaissance high_renaissance   
artist                                                           
agnolo-bronzino                            13               18   
andrea-del-sarto                            8                9   
andrea-mantegna                             4                6   
andrea-solario                              1                1   
cima-da-conegliano                          2                2   
correggio                                   2                1   

template                                                            noun  \
prediction         mannerism_late_renaissance mannerism_late_renaissance   
artist                                                                     
agnolo-bronzino                             0                         18   
andrea-del-sarto                            1                         10   
andrea-mantegna                             0                          6   
andrea-solario                              1                          2   
cima-da-conegliano                          0                          2   
correggio                                   1                          2   

template                      style                             
prediction         high_renaissance mannerism_late_renaissance  
artist                                                          
agnolo-bronzino                  10                          8  
andrea-del-sarto                  3                          7  
andrea-mantegna                   4                          2  
andrea-solario                    1                          1  
cima-da-conegliano                0                          2  
correggio                         0                          2


Saved project2_prompt_agreement.csv


In [6]:

# Controlled follow-up: matched grammatical structures
controlled_prompts = {
    "style": (
        "a painting in the High Renaissance style",
        "a painting in the Mannerist style"
    ),
    "noun": (
        "a High Renaissance painting",
        "a Mannerist painting"
    ),
    "artwork": (
        "an artwork from the High Renaissance period",
        "an artwork from the Mannerist period"
    ),
    "characteristic": (
        "a painting characteristic of the High Renaissance",
        "a painting characteristic of Mannerism"
    ),
    "artistic_style": (
        "the artistic style of the High Renaissance",
        "the artistic style of Mannerism"
    )
}

prompts = [
    p for pair in controlled_prompts.values() for p in pair
]

text_inputs = processor(
    text=prompts,
    return_tensors="pt",
    padding=True,
    truncation=True
).to(device)

with torch.inference_mode():
    out = model.text_model(
        input_ids=text_inputs["input_ids"],
        attention_mask=text_inputs["attention_mask"]
    )
    text_features2 = F.normalize(
        model.text_projection(out.pooler_output), dim=-1
    )

similarities2 = (
    image_features.to(device) @ text_features2.T
).cpu().numpy()

controlled_results = []

for j, template in enumerate(controlled_prompts):
    for i, item in enumerate(sample_balanced):
        high_score = float(similarities2[i, 2*j])
        mannerism_score = float(similarities2[i, 2*j+1])
        prediction = (
            "high_renaissance"
            if high_score > mannerism_score
            else "mannerism_late_renaissance"
        )
        controlled_results.append({
            "image_id": item["image_id"],
            "artist": item["artist"],
            "true_style": item["style"],
            "template": template,
            "prediction": prediction,
            "correct": prediction == item["style"],
            "high_score": high_score,
            "mannerism_score": mannerism_score
        })

controlled_df = pd.DataFrame(controlled_results)

print("=== CONTROLLED PROMPT ACCURACY ===")
print(
    controlled_df.groupby("template")["correct"]
    .agg(["mean", "sum", "count"]).to_string()
)

controlled_pivot = controlled_df.pivot(
    index="image_id", columns="template", values="prediction"
)
print("\nImages changing prediction across controlled templates:",
      int((controlled_pivot.nunique(axis=1) > 1).sum()), "of",
      len(controlled_pivot))

print("\nPredicted class counts:")
for template, group in controlled_df.groupby("template", sort=False):
    print(template, group["prediction"].value_counts().to_dict())

controlled_df.to_csv(
    "project2_controlled_prompt_robustness.csv", index=False
)
print("\nSaved controlled-prompt results.")


=== CONTROLLED PROMPT ACCURACY ===
                 mean  sum  count
template                         
artistic_style  0.500   20     40
artwork         0.500   20     40
characteristic  0.475   19     40
noun            0.500   20     40
style           0.450   18     40

Images changing prediction across controlled templates: 40 of 40

Predicted class counts:
style {'mannerism_late_renaissance': 22, 'high_renaissance': 18}
noun {'mannerism_late_renaissance': 40}
artwork {'mannerism_late_renaissance': 30, 'high_renaissance': 10}
characteristic {'high_renaissance': 37, 'mannerism_late_renaissance': 3}
artistic_style {'high_renaissance': 40}

Saved controlled-prompt results.


In [5]:

# 1. Accuracy by artist and prompt wording
print("=== ACCURACY BY ARTIST AND PROMPT ===")
artist_accuracy = results_df.pivot_table(
    index="artist",
    columns="template",
    values="correct",
    aggfunc="mean"
)
display(artist_accuracy.round(3))

# 2. Which style does each prompt tend to predict?
print("\n=== PREDICTED CLASS COUNTS ===")
for template, group in results_df.groupby("template", sort=False):
    print(f"\n{template}")
    print(group["prediction"].value_counts().to_dict())

# 3. Accuracy separately for each dataset label
print("\n=== ACCURACY BY TRUE STYLE ===")
display(
    results_df.pivot_table(
        index="true_style",
        columns="template",
        values="correct",
        aggfunc="mean"
    ).round(3)
)

# 4. How close are the competing text similarities?
results_df["score_margin"] = (
    results_df["high_score"] - results_df["mannerism_score"]
).abs()

print("\n=== ABSOLUTE SIMILARITY MARGIN ===")
print(
    results_df.groupby("template")["score_margin"]
    .agg(["mean", "median", "min", "max"])
    .round(4)
)

# 5. Save a compact summary for the project report
summary = results_df.groupby("template").agg(
    accuracy=("correct", "mean"),
    correct=("correct", "sum"),
    mean_absolute_margin=("score_margin", "mean"),
    median_absolute_margin=("score_margin", "median")
).reset_index()

summary.to_csv("project2_prompt_robustness_summary.csv", index=False)
print("\nSaved project2_prompt_robustness_summary.csv")


=== ACCURACY BY ARTIST AND PROMPT ===


template,artwork,basic,example,historical,visual_style
artist,,,,,
agnolo-bronzino,0.944,0.444,1.0,0.444,0.0
andrea-del-sarto,0.000,0.200,0.0,0.500,1.0
andrea-mantegna,0.000,0.500,0.0,0.500,1.0
andrea-solario,0.000,0.500,0.0,0.500,1.0
cima-da-conegliano,0.000,1.000,0.0,0.000,1.0
correggio,1.000,1.000,1.0,1.000,0.0



=== PREDICTED CLASS COUNTS ===

basic
{'mannerism_late_renaissance': 22, 'high_renaissance': 18}

artwork
{'mannerism_late_renaissance': 39, 'high_renaissance': 1}

example
{'mannerism_late_renaissance': 40}

historical
{'mannerism_late_renaissance': 21, 'high_renaissance': 19}

visual_style
{'high_renaissance': 40}

=== ACCURACY BY TRUE STYLE ===


template,artwork,basic,example,historical,visual_style
true_style,,,,,
high_renaissance,0.00,0.4,0.0,0.45,1.0
mannerism_late_renaissance,0.95,0.5,1.0,0.50,0.0



=== ABSOLUTE SIMILARITY MARGIN ===
                mean  median     min     max
template                                    
artwork       0.0182  0.0169  0.0009  0.0343
basic         0.0092  0.0088  0.0003  0.0205
example       0.0255  0.0246  0.0056  0.0409
historical    0.0086  0.0086  0.0007  0.0183
visual_style  0.0380  0.0396  0.0104  0.0633

Saved project2_prompt_robustness_summary.csv


In [4]:

import torch
import torch.nn.functional as F
import pandas as pd

# Reuse the model, processor, sample_balanced and prompt_pairs
model.eval()

device = next(model.parameters()).device
images = [x["image"].convert("RGB") for x in sample_balanced]
labels = [x["style"] for x in sample_balanced]

all_prompts = [
    prompt
    for pair in prompt_pairs.values()
    for prompt in pair
]

# Text features: encoder output -> CLIP projection -> normalize
text_inputs = processor(
    text=all_prompts,
    return_tensors="pt",
    padding=True,
    truncation=True
).to(device)

with torch.inference_mode():
    text_output = model.text_model(
        input_ids=text_inputs["input_ids"],
        attention_mask=text_inputs["attention_mask"]
    )
    text_features = model.text_projection(text_output.pooler_output)
    text_features = F.normalize(text_features, dim=-1)

# Image features in small batches for CPU/RAM safety
image_features_list = []

with torch.inference_mode():
    for start in range(0, len(images), 4):
        batch = images[start:start + 4]
        inputs = processor(images=batch, return_tensors="pt").to(device)

        vision_output = model.vision_model(
            pixel_values=inputs["pixel_values"]
        )
        features = model.visual_projection(vision_output.pooler_output)
        features = F.normalize(features, dim=-1)
        image_features_list.append(features.cpu())

image_features = torch.cat(image_features_list, dim=0)

print("Image feature shape:", tuple(image_features.shape))
print("Text feature shape:", tuple(text_features.shape))

# Compare each image against each matched prompt pair
similarities = (
    image_features.to(device) @ text_features.T
).cpu().numpy()

results = []

for template_index, template in enumerate(prompt_pairs):
    high_col = template_index * 2
    mannerism_col = high_col + 1

    for i, item in enumerate(sample_balanced):
        high_score = float(similarities[i, high_col])
        mannerism_score = float(similarities[i, mannerism_col])

        prediction = (
            "high_renaissance"
            if high_score > mannerism_score
            else "mannerism_late_renaissance"
        )

        results.append({
            "image_id": item["image_id"],
            "artist": item["artist"],
            "true_style": labels[i],
            "template": template,
            "high_score": high_score,
            "mannerism_score": mannerism_score,
            "prediction": prediction,
            "correct": prediction == labels[i]
        })

results_df = pd.DataFrame(results)

print("\nAccuracy by prompt template:")
print(
    results_df.groupby("template")["correct"]
    .agg(["mean", "sum", "count"])
    .to_string()
)

prediction_pivot = results_df.pivot(
    index="image_id", columns="template", values="prediction"
)
flip_counts = prediction_pivot.nunique(axis=1)

print("\nImages changing prediction across templates:",
      int((flip_counts > 1).sum()), "of", len(flip_counts))
print("Mean distinct predictions per image:",
      round(flip_counts.mean(), 3))

display(results_df.head(10))
results_df.to_csv("project2_prompt_robustness_results.csv", index=False)
print("\nSaved results CSV.")


Image feature shape: (40, 512)
Text feature shape: (10, 512)

Accuracy by prompt template:
               mean  sum  count
template                       
artwork       0.475   19     40
basic         0.450   18     40
example       0.500   20     40
historical    0.475   19     40
visual_style  0.500   20     40

Images changing prediction across templates: 40 of 40
Mean distinct predictions per image: 2.0


,image_id,artist,true_style,template,high_score,mannerism_score,prediction,correct
0,High_Renaissance/andrea-del-sarto_baptism-of-t...,andrea-del-sarto,high_renaissance,basic,0.274053,0.280327,mannerism_late_renaissance,False
1,High_Renaissance/andrea-del-sarto_head-of-a-ch...,andrea-del-sarto,high_renaissance,basic,0.272228,0.289021,mannerism_late_renaissance,False
2,High_Renaissance/andrea-del-sarto_stories-of-j...,andrea-del-sarto,high_renaissance,basic,0.299131,0.283584,high_renaissance,True
3,High_Renaissance/andrea-del-sarto_portrait-of-...,andrea-del-sarto,high_renaissance,basic,0.296038,0.306242,mannerism_late_renaissance,False
4,High_Renaissance/andrea-del-sarto_charity.jpg,andrea-del-sarto,high_renaissance,basic,0.291945,0.304630,mannerism_late_renaissance,False
5,High_Renaissance/andrea-del-sarto_madonna-and-...,andrea-del-sarto,high_renaissance,basic,0.285340,0.288018,mannerism_late_renaissance,False
6,High_Renaissance/andrea-mantegna_judith-with-t...,andrea-mantegna,high_renaissance,basic,0.262804,0.271541,mannerism_late_renaissance,False
7,High_Renaissance/andrea-mantegna_saint-jerome-...,andrea-mantegna,high_renaissance,basic,0.272328,0.277612,mannerism_late_renaissance,False
8,High_Renaissance/andrea-del-sarto_lucrezia-di-...,andrea-del-sarto,high_renaissance,basic,0.287509,0.300540,mannerism_late_renaissance,False
9,High_Renaissance/andrea-del-sarto_st-john-the-...,andrea-del-sarto,high_renaissance,basic,0.300863,0.307413,mannerism_late_renaissance,False



Saved results CSV.


In [3]:

import torch
import pandas as pd
from transformers import CLIPModel, CLIPProcessor
from collections import Counter

MODEL_NAME = "openai/clip-vit-base-patch32"
device = "cuda" if torch.cuda.is_available() else "cpu"

print("Device:", device)
print("Loading CLIP...")

processor = CLIPProcessor.from_pretrained(MODEL_NAME)
model = CLIPModel.from_pretrained(MODEL_NAME).to(device)
model.eval()

# Five matched prompt pairs: wording varies, artwork stays fixed.
prompt_pairs = {
    "basic": (
        "a painting in the High Renaissance style",
        "a painting in the Mannerism style"
    ),
    "artwork": (
        "a High Renaissance artwork",
        "a Mannerist artwork"
    ),
    "example": (
        "an example of High Renaissance art",
        "an example of Mannerist art"
    ),
    "historical": (
        "a painting created during the High Renaissance",
        "a painting created during the Mannerist period"
    ),
    "visual_style": (
        "the visual style of the High Renaissance",
        "the visual style of Mannerism"
    )
}

images = [x["image"].convert("RGB") for x in sample_balanced]
labels = [x["style"] for x in sample_balanced]

all_prompts = [
    prompt
    for pair in prompt_pairs.values()
    for prompt in pair
]

text_inputs = processor(
    text=all_prompts,
    return_tensors="pt",
    padding=True,
    truncation=True
).to(device)

with torch.inference_mode():
    text_features = model.get_text_features(**text_inputs)
    text_features = torch.nn.functional.normalize(
        text_features, dim=-1
    )

# Process images in small batches to limit memory use.
image_features_list = []

with torch.inference_mode():
    for start in range(0, len(images), 4):
        batch = images[start:start + 4]
        inputs = processor(
            images=batch,
            return_tensors="pt"
        ).to(device)

        features = model.get_image_features(
            pixel_values=inputs["pixel_values"]
        )
        features = torch.nn.functional.normalize(
            features, dim=-1
        )
        image_features_list.append(features.cpu())

image_features = torch.cat(image_features_list, dim=0)

# Cosine similarities; each matched pair competes on the same image.
similarities = image_features.to(device) @ text_features.T
similarities = similarities.cpu().numpy()

results = []

for template_index, (template, pair) in enumerate(prompt_pairs.items()):
    high_col = template_index * 2
    mannerism_col = high_col + 1

    for i, item in enumerate(sample_balanced):
        high_score = float(similarities[i, high_col])
        mannerism_score = float(similarities[i, mannerism_col])

        prediction = (
            "high_renaissance"
            if high_score > mannerism_score
            else "mannerism_late_renaissance"
        )

        results.append({
            "image_id": item["image_id"],
            "artist": item["artist"],
            "true_style": labels[i],
            "template": template,
            "high_score": high_score,
            "mannerism_score": mannerism_score,
            "prediction": prediction,
            "correct": prediction == labels[i]
        })

results_df = pd.DataFrame(results)

print("\nAccuracy by prompt template:")
print(
    results_df.groupby("template")["correct"]
    .agg(["mean", "sum", "count"])
    .to_string()
)

print("\nOverall accuracy across image-template pairs:",
      round(results_df["correct"].mean(), 3))

# Fraction of images whose predicted class changes across templates.
prediction_pivot = results_df.pivot(
    index="image_id",
    columns="template",
    values="prediction"
)

flip_counts = prediction_pivot.nunique(axis=1)

print("\nImages with at least one prediction change:",
      int((flip_counts > 1).sum()), "of", len(flip_counts))

print("Mean distinct predictions per image:",
      round(flip_counts.mean(), 3))

print("\nResults table:")
display(results_df.head(10))

results_df.to_csv("project2_prompt_robustness_results.csv", index=False)
print("\nSaved project2_prompt_robustness_results.csv")


Device: cpu
Loading CLIP...


preprocessor_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/4.19k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/592 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/862k [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  605MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

AttributeError: 'BaseModelOutputWithPooling' object has no attribute 'norm'

In [2]:

from collections import Counter

print("Sample size:", len(sample_balanced))
print("\nStyle counts:", dict(Counter(
    x["style"] for x in sample_balanced
)))

print("\nArtist/style combinations:")
for item in sample_balanced:
    print(
        item["artist"],
        "|",
        item["style"],
        "|",
        item["title"],
        "|",
        item["image"].size if hasattr(item["image"], "size") else type(item["image"])
    )


Sample size: 40

Style counts: {'high_renaissance': 20, 'mannerism_late_renaissance': 20}

Artist/style combinations:
andrea-del-sarto | high_renaissance | andrea-del-sarto_baptism-of-the-people-1517.jpg | (997, 801)
andrea-del-sarto | high_renaissance | andrea-del-sarto_head-of-a-child.jpg | (1024, 1536)
andrea-del-sarto | high_renaissance | andrea-del-sarto_stories-of-joseph.jpg | (988, 672)
andrea-del-sarto | high_renaissance | andrea-del-sarto_portrait-of-baccio-bandinelli.jpg | (900, 1256)
andrea-del-sarto | high_renaissance | andrea-del-sarto_charity.jpg | (850, 1167)
andrea-del-sarto | high_renaissance | andrea-del-sarto_madonna-and-child-with-st-john-the-baptist-1.jpg | (523, 640)
andrea-mantegna | high_renaissance | andrea-mantegna_judith-with-the-head-of-holofernes-1495.jpg | (335, 500)
andrea-mantegna | high_renaissance | andrea-mantegna_saint-jerome-reading-with-the-lion-1500.jpg | (1455, 1910)
andrea-del-sarto | high_renaissance | andrea-del-sarto_lucrezia-di-baccio-del-fe

In [1]:

from datasets import load_dataset
from collections import Counter

TARGET_STYLES = {
    "high_renaissance",
    "mannerism_late_renaissance"
}

ds_shuffled = load_dataset(
    "Shuaimyself/MMArt",
    split="train",
    streaming=True
).shuffle(seed=42, buffer_size=500)

sample_balanced = []
counts_balanced = Counter()

for item in ds_shuffled:
    style = item.get("style")

    if style not in TARGET_STYLES:
        continue

    if counts_balanced[style] >= 20:
        continue

    if item.get("image") is None:
        continue

    sample_balanced.append({
        "image_id": item.get("image_id"),
        "title": item.get("title"),
        "artist": item.get("artist"),
        "style": style,
        "image": item.get("image")
    })
    counts_balanced[style] += 1

    if all(counts_balanced[s] >= 20 for s in TARGET_STYLES):
        break

print("Total:", len(sample_balanced))
print("Style counts:", dict(counts_balanced))
print("Artists per style:")
for style in TARGET_STYLES:
    artists = {
        x["artist"] for x in sample_balanced
        if x["style"] == style
    }
    print(style, len(artists), sorted(artists))


README.md:   0%|          | 0.00/5.53k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/22 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/22 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/PIL/Image.py:3452: DecompressionBombWarning: Image size (99962094 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/PIL/Image.py:3452: DecompressionBombWarning: Image size (107327830 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(


Total: 40
Style counts: {'high_renaissance': 20, 'mannerism_late_renaissance': 20}
Artists per style:
mannerism_late_renaissance 2 ['agnolo-bronzino', 'correggio']
high_renaissance 4 ['andrea-del-sarto', 'andrea-mantegna', 'andrea-solario', 'cima-da-conegliano']


In [ ]:

from datasets import load_dataset

ds_test = load_dataset(
    "Shuaimyself/MMArt",
    split="train",
    streaming=True
)

print("Shuffle available:", hasattr(ds_test, "shuffle"))
print("Dataset type:", type(ds_test).__name__)


Resolving data files:   0%|          | 0/22 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/22 [00:00<?, ?it/s]

Shuffle available: True
Dataset type: IterableDataset


In [ ]:

import pandas as pd

sample_df = pd.DataFrame([
    {
        "title": item["title"],
        "artist": item["artist"],
        "style": item["style"]
    }
    for item in sample
])

print(sample_df.groupby(["style", "artist"]).size().to_string())


style                       artist          
high_renaissance            andrea-del-sarto    20
mannerism_late_renaissance  agnolo-bronzino     20


In [ ]:

from datasets import load_dataset
from collections import Counter

TARGET_STYLES = {
    "high_renaissance",
    "mannerism_late_renaissance"
}

sample = []
counts = Counter()

ds = load_dataset(
    "Shuaimyself/MMArt",
    split="train",
    streaming=True
)

for item in ds:
    style = item.get("style")

    if style not in TARGET_STYLES:
        continue

    if counts[style] >= 20:
        continue

    image = item.get("image")
    if image is None:
        continue

    sample.append({
        "image_id": item.get("image_id"),
        "title": item.get("title"),
        "artist": item.get("artist"),
        "style": style,
        "image": image
    })
    counts[style] += 1

    if all(counts[s] >= 20 for s in TARGET_STYLES):
        break

print("Collected:", len(sample))
print("Counts:", dict(counts))
print("Artists represented:", len(set(x["artist"] for x in sample)))


Resolving data files:   0%|          | 0/22 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/22 [00:00<?, ?it/s]

Collected: 40
Counts: {'high_renaissance': 20, 'mannerism_late_renaissance': 20}
Artists represented: 2


In [ ]:

from collections import Counter
from datasets import load_dataset

TARGET_STYLES = {
    "high_renaissance",
    "mannerism_late_renaissance"
}

ds = load_dataset(
    "Shuaimyself/MMArt",
    split="train",
    streaming=True
)

style_counts = Counter()

for item in ds:
    style = item.get("style")
    if style in TARGET_STYLES:
        style_counts[style] += 1

print("Target style counts:")
for style, count in style_counts.items():
    print(f"{style}: {count}")


Resolving data files:   0%|          | 0/22 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/22 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/PIL/Image.py:3452: DecompressionBombWarning: Image size (107327830 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/PIL/Image.py:3452: DecompressionBombWarning: Image size (99962094 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(


Target style counts:
high_renaissance: 1337
mannerism_late_renaissance: 1267


In [ ]:

from datasets import load_dataset

ds = load_dataset(
    "Shuaimyself/MMArt",
    split="train",
    streaming=True
)

first_item = next(iter(ds))

print("Available columns:")
print(list(first_item.keys()))

print("\nStyle:", first_item.get("style"))
print("Title:", first_item.get("title"))
print("Artist:", first_item.get("artist"))
print("Image available:", first_item.get("image") is not None)


README.md:   0%|          | 0.00/5.53k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/22 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/22 [00:00<?, ?it/s]

Available columns:
['image_id', 'title', 'artist', 'style', 'date', 'e_narrative', 'e_formal', 'e_emotional', 'e_historical', 'dominant_emotion', 'artemis_coverage', 'rag_sim', 'n_perspectives', 'e_unified', 'image']

Style: abstract_expressionism
Title: aaron-siskind_acolman-1-1955.jpg
Artist: aaron-siskind
Image available: True


In [ ]:

!pip -q install datasets


In [ ]:

import torch
from transformers import CLIPModel, CLIPProcessor

MODEL_NAME = "openai/clip-vit-base-patch32"
device = "cuda" if torch.cuda.is_available() else "cpu"

model = CLIPModel.from_pretrained(MODEL_NAME).to(device)
processor = CLIPProcessor.from_pretrained(MODEL_NAME)
model.eval()

print("Model:", MODEL_NAME)
print("Device:", device)
print("Model loaded successfully.")


config.json:   0%|          | 0.00/4.19k [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  605MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  605MB            

...

vocab.json:   0%|          | 0.00/862k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/525k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.22M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

Model: openai/clip-vit-base-patch32
Device: cpu
Model loaded successfully.


In [ ]:

import transformers
import pandas
import sklearn
import torch

print("Libraries installed successfully!")
print("Transformers:", transformers.__version__)
print("PyTorch:", torch.__version__)


Libraries installed successfully!
Transformers: 5.18.0
PyTorch: 2.11.0+cpu


In [ ]:

!pip -q install transformers pillow pandas scikit-learn

